# ArbGraph: Conflict-Aware Evidence Arbitration for Reliable Long-Form RAG

**Paper implementation** — faithful reproduction of the original codebase.  
Compatible with **Google Colab** (T4/A100) and **Kaggle** (P100/T4).

### Pipeline
```
Dataset → Retrieval (Wikipedia) → Atomization → Claim Alignment
       → Evidence Graph → Conflict Arbitration → Long-form Generation
```

### Supported dataset formats
| Dataset | Format | Question key | ID key |
|---------|--------|-------------|--------|
| LongFact | JSON list | `Question` | `id` |
| TriviaQA | JSON (data list) | `question` | `question_id` |
| NQ-Open | JSONL | `question` | `id` |
| PopQA | TSV | `question` | `id` |
| Custom CSV/JSON | configurable | configurable | configurable |

## Cell 1 — Install dependencies

In [ ]:
# ── Install all required packages ──────────────────────────────────────────
import subprocess, sys

packages = [
    "torch>=2.0",
    "transformers>=4.40",
    "sentence-transformers>=2.2",
    "networkx>=3.0",
    "numpy>=1.23",
    "tqdm>=4.60",
    "requests>=2.28",
    "accelerate>=0.26",
    "bitsandbytes>=0.41",   # 4-bit quantisation for large GPUs
    "scikit-learn>=1.2",
    "pandas>=1.5",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✓ All packages installed")

## Cell 2 — Clone ArbGraph repo

In [ ]:
import os, sys

REPO_DIR = "/content/ArbGraph"  # change to /kaggle/working/ArbGraph on Kaggle

if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/1212Judy/ArbGraph {REPO_DIR}")
    print("✓ Repo cloned")
else:
    os.system(f"cd {REPO_DIR} && git pull")
    print("✓ Repo up to date")

# Add to Python path so all original modules are importable without modification
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.listdir(REPO_DIR)

## Cell 3 — GPU check

In [ ]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM     : {total:.1f} GB")
    if total < 12:
        print("⚠  < 12 GB VRAM — will use 4-bit quantisation (bitsandbytes)")
    elif total < 20:
        print("✓  Sufficient VRAM for fp16")
    else:
        print("✓  Large GPU — can run bf16")

## Cell 4 — Original HFAdapter (exact copy from repo)

This cell reproduces `local_llm/hf_adapter.py` verbatim and adds **4-bit quantisation** as a drop-in option for smaller GPUs. The class interface is identical to the original — no downstream code changes required.

In [ ]:
# ── Exact copy of local_llm/hf_adapter.py + 4-bit option ──────────────────
# The only addition is the `load_in_4bit` parameter for T4/P100 GPUs.
# Everything else is byte-for-byte identical to the original.

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


class HFAdapter:
    """
    Minimal HuggingFace-based LLM adapter.
    Provides a unified generate() interface for the ArbGraph pipeline.
    
    Original: local_llm/hf_adapter.py
    Addition: load_in_4bit flag for GPUs with < 20 GB VRAM.
    """

    def __init__(self, model_name_or_path: str, load_in_4bit: bool = False):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name_or_path,
            use_fast=True
        )

        if load_in_4bit and torch.cuda.is_available():
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name_or_path,
                device_map="auto",
                quantization_config=bnb_config,
            ).eval()
        else:
            # ── ORIGINAL code, unchanged ────────────────────────────────
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name_or_path,
                device_map="auto",
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            ).eval()

    def generate(
        self,
        prompt: str,
        max_new_tokens: int = 400,
        temperature: float = 0.0,
    ) -> str:
        # ── ORIGINAL generate() — unchanged ────────────────────────────
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt"
        ).to(self.model.device)

        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "use_cache": True,
            "pad_token_id": self.tokenizer.eos_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
        }

        if temperature and temperature > 0:
            generation_kwargs["do_sample"] = True
            generation_kwargs["temperature"] = temperature
        else:
            generation_kwargs["do_sample"] = False

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **generation_kwargs
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )


print("✓ HFAdapter defined")

## Cell 5 — Config: model, paths, dataset

In [ ]:
import os

# ──────────────────────────────────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────────────────────────────────
# Default from paper: Qwen/Qwen3-4B-Instruct
# Alternatives (uncomment as needed):
#   "Qwen/Qwen2.5-3B-Instruct"         # smaller, faster
#   "Qwen/Qwen3-8B-Instruct"            # better quality, needs ~16 GB
#   "mistralai/Mistral-7B-Instruct-v0.3"  # good general alternative
#   "google/gemma-2-2b-it"              # small, fast
#   "meta-llama/Meta-Llama-3.1-8B-Instruct"  # needs HF token

MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen3-4B-Instruct")

# ──────────────────────────────────────────────────────────────────────────
# GPU MEMORY — set True for T4 (15 GB) or P100 (16 GB)
# ──────────────────────────────────────────────────────────────────────────
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
LOAD_IN_4BIT = vram_gb < 20   # auto-enable 4-bit on T4/P100

# ──────────────────────────────────────────────────────────────────────────
# DATASET CONFIG
# ──────────────────────────────────────────────────────────────────────────
# Choose one preset or set DATASET_FORMAT = "custom" and configure manually.

DATASET_PRESET = "longfact"   # options: longfact | triviaqa | nq_open | popqa | custom

DATASET_PRESETS = {
    "longfact": {
        "input_file": "/content/data/longfact/longfact_test.json",
        "output_file": "/content/outputs/longfact/arbgraph_predictions.jsonl",
        "question_key": "Question",
        "id_key": "id",
        "file_format": "json",          # json | jsonl | csv | tsv
        "data_path": None,              # None = root list; string = nested key e.g. "data.items"
        "max_docs": 5,
    },
    "triviaqa": {
        "input_file": "/content/data/triviaqa/test.json",
        "output_file": "/content/outputs/triviaqa/arbgraph_predictions.jsonl",
        "question_key": "question",
        "id_key": "question_id",
        "file_format": "json",
        "data_path": "data",            # TriviaQA wraps list under {"data": [...]}
        "max_docs": 5,
    },
    "nq_open": {
        "input_file": "/content/data/nq_open/test.jsonl",
        "output_file": "/content/outputs/nq_open/arbgraph_predictions.jsonl",
        "question_key": "question",
        "id_key": "id",
        "file_format": "jsonl",
        "data_path": None,
        "max_docs": 5,
    },
    "popqa": {
        "input_file": "/content/data/popqa/test.tsv",
        "output_file": "/content/outputs/popqa/arbgraph_predictions.jsonl",
        "question_key": "question",
        "id_key": "id",
        "file_format": "tsv",
        "data_path": None,
        "max_docs": 3,
    },
    "custom": {
        "input_file": "/content/data/my_dataset.json",   # ← your file
        "output_file": "/content/outputs/custom/arbgraph_predictions.jsonl",
        "question_key": "question",    # ← column/key name for the question text
        "id_key": "id",                # ← column/key name for the unique ID
        "file_format": "json",         # json | jsonl | csv | tsv
        "data_path": None,
        "max_docs": 5,
    },
}

CFG = DATASET_PRESETS[DATASET_PRESET]

# ──────────────────────────────────────────────────────────────────────────
# ARBITRATOR HYPERPARAMS (paper defaults)
# ──────────────────────────────────────────────────────────────────────────
MAX_ROUNDS          = 2     # arbitration rounds (paper uses 2)
ACCEPT_THRESHOLD    = 0.3   # min probability to keep a claim
ARBITRATION_BUDGET  = 3     # max conflict pairs resolved per round
STEP_SIZE           = 0.8   # η in logit update

# ──────────────────────────────────────────────────────────────────────────
# EVIDENCE GRAPH HYPERPARAMS (paper defaults)
# ──────────────────────────────────────────────────────────────────────────
QUERY_RELEVANCE_THRESHOLD  = 0.3
PAIR_SIMILARITY_THRESHOLD  = 0.75
MAX_SUPPORT_EDGES          = 60

# ──────────────────────────────────────────────────────────────────────────
# NODE ALIGNER
# ──────────────────────────────────────────────────────────────────────────
ENCODER_NAME        = "BAAI/bge-large-en-v1.5"   # paper default
SIMILARITY_THRESHOLD = 0.88

print(f"Model           : {MODEL_NAME}")
print(f"4-bit quant     : {LOAD_IN_4BIT}  (VRAM = {vram_gb:.1f} GB)")
print(f"Dataset preset  : {DATASET_PRESET}")
print(f"Input file      : {CFG['input_file']}")
print(f"Output file     : {CFG['output_file']}")

## Cell 6 — Universal dataset loader

In [ ]:
import json, csv
import pandas as pd
from typing import List, Dict


def load_dataset(cfg: dict) -> List[Dict]:
    """
    Load any dataset into a unified list of {"id": ..., "Question": ...} dicts.
    The pipeline always reads ["Question"] and ["id"] — this normalises all
    dataset formats to match the original LongFact schema used by run_arbgraph.py.
    """
    fmt   = cfg["file_format"]
    fpath = cfg["input_file"]
    qkey  = cfg["question_key"]
    ikey  = cfg["id_key"]
    dpath = cfg.get("data_path")   # optional nested key

    # ── Read raw ──────────────────────────────────────────────────────────
    if fmt == "json":
        with open(fpath, "r", encoding="utf-8") as f:
            raw = json.load(f)
        if dpath:
            for key in dpath.split("."):
                raw = raw[key]

    elif fmt == "jsonl":
        raw = []
        with open(fpath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    raw.append(json.loads(line))

    elif fmt in {"csv", "tsv"}:
        sep = "," if fmt == "csv" else "\t"
        df  = pd.read_csv(fpath, sep=sep)
        raw = df.to_dict(orient="records")

    else:
        raise ValueError(f"Unsupported format: {fmt}")

    # ── Normalise to {id, Question} ───────────────────────────────────────
    questions = []
    for i, item in enumerate(raw):
        q_text = item.get(qkey, "")
        q_id   = item.get(ikey, f"q_{i}")
        if not q_text:
            continue
        # Preserve all original fields + add normalised keys
        record = dict(item)
        record["Question"] = str(q_text).strip()
        record["id"]       = str(q_id)
        questions.append(record)

    return questions


# ── Demo: create a tiny sample dataset if the real file doesn't exist ──────
if not os.path.exists(CFG["input_file"]):
    print(f"⚠  {CFG['input_file']} not found — creating sample data for testing.")
    sample_dir = os.path.dirname(CFG["input_file"])
    os.makedirs(sample_dir, exist_ok=True)

    sample_questions = [
        {"id": "q001", "Question": "What are the main causes of World War I?"},
        {"id": "q002", "Question": "How does the CRISPR-Cas9 gene editing system work?"},
        {"id": "q003", "Question": "What were the key factors behind the 2008 financial crisis?"},
    ]

    with open(CFG["input_file"], "w", encoding="utf-8") as f:
        json.dump(sample_questions, f, indent=2)
    print(f"✓ Sample dataset written to {CFG['input_file']}")

questions = load_dataset(CFG)
print(f"✓ Loaded {len(questions)} questions from {CFG['input_file']}")
print(f"  First question: {questions[0]['Question'][:80]}")

## Cell 7 — Load model and initialise all pipeline components

This uses the **exact** classes from the cloned repo — no modifications.

In [ ]:
import sys, os
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Import original modules from the cloned repo ───────────────────────────
from atomization        import ClaimExtractor
from retrieval.retriever import WikipediaRetriever
from claim_alignment    import NodeAligner
from evidence_graph     import EvidenceGraphBuilder
from conflict_arbitration import ArbGraphArbitrator
from longform_generation import AnswerGenerator

print("Loading model — this may take 5-10 minutes on first run (downloads weights)...")
llm = HFAdapter(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT)
print(f"✓ LLM loaded: {MODEL_NAME}")

# ── Initialise all pipeline components (original constructors, unchanged) ──
retriever    = WikipediaRetriever()
extractor    = ClaimExtractor(llm)
aligner      = NodeAligner(
    llm_engine=llm,
    encoder_name=ENCODER_NAME,
    similarity_threshold=SIMILARITY_THRESHOLD,
)
graph_builder = EvidenceGraphBuilder(
    llm,
    embedding_model=ENCODER_NAME,
)
arbitrator   = ArbGraphArbitrator(
    llm=llm,
    max_rounds=MAX_ROUNDS,
    accept_threshold=ACCEPT_THRESHOLD,
    arbitration_budget=ARBITRATION_BUDGET,
    step_size=STEP_SIZE,
)
generator    = AnswerGenerator(llm)

# Pack exactly as run_arbgraph.py does
components = {
    "retriever":     retriever,
    "extractor":     extractor,
    "aligner":       aligner,
    "graph_builder": graph_builder,
    "arbitrator":    arbitrator,
    "generator":     generator,
}

print("✓ All pipeline components initialised")

## Cell 8 — Core pipeline functions

Exact copies of `extract_claims_from_documents`, `process_single_question`, and `load_checkpoint` from `run_arbgraph.py`. Zero changes.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EXACT COPY of run_arbgraph.py — functions unchanged
# ═══════════════════════════════════════════════════════════════════════════

import json
from typing import Dict


def extract_claims_from_documents(extractor, documents):
    claims = []

    for doc in documents:
        text = doc.text
        chunk_size = 2500
        step = chunk_size - 200
        chunks = [
            text[i:i + chunk_size]
            for i in range(0, len(text), step)
        ]

        for chunk_idx, chunk in enumerate(chunks):
            chunk_claims = extractor.extract(
                text=chunk,
                source_id=doc.id,
                doc_id=doc.id,
                doc_title=getattr(doc, "title", None),
            )
            claims.extend(chunk_claims)

    return claims


def process_single_question(
    question_data: Dict,
    components: Dict,
    output_file: str,
    max_docs: int = 5,
):
    retriever     = components["retriever"]
    extractor     = components["extractor"]
    aligner       = components["aligner"]
    graph_builder = components["graph_builder"]
    arbitrator    = components["arbitrator"]
    generator     = components["generator"]

    query = question_data["Question"]
    qid   = question_data["id"]

    result = {
        "id": qid,
        "question": query,
        "documents": [],
        "arbgraph_meta": {
            "rounds": 0,
            "validated_claims": 0,
            "suppressed_claims": 0,
            "retrieval_status": "failed",
            "status": "init",
            "error_stage": None,
        },
        "answer": ""
    }

    try:
        docs = retriever.retrieve(query, max_docs=max_docs)
        if not docs:
            result["arbgraph_meta"]["status"] = "no_documents"
        else:
            result["arbgraph_meta"]["retrieval_status"] = "success"

            for idx, doc in enumerate(docs, 1):
                result["documents"].append({
                    "id": idx,
                    "title": getattr(doc, "title", ""),
                    "url": getattr(doc, "url", ""),
                    "source": "Wikipedia",
                    "context": getattr(doc, "text", "")[:3000],
                })

            raw_claims = extract_claims_from_documents(extractor, docs)
            if not raw_claims:
                result["arbgraph_meta"]["status"] = "no_raw_claims"
                result["arbgraph_meta"]["error_stage"] = "claim_extraction"
            else:
                canonical_claims = aligner.align_and_merge(raw_claims)
                if not canonical_claims:
                    result["arbgraph_meta"]["status"] = "no_canonical_claims"
                    result["arbgraph_meta"]["error_stage"] = "claim_alignment"
                else:
                    graph = graph_builder.build(canonical_claims, query=query)

                    arbitration_result = arbitrator.arbitrate(graph, query)

                    validated_claims  = arbitration_result.get("validated_claims", [])
                    suppressed_claims = arbitration_result.get("suppressed_claims", [])

                    result["arbgraph_meta"]["validated_claims"]  = len(validated_claims)
                    result["arbgraph_meta"]["suppressed_claims"] = len(suppressed_claims)

                    history = arbitration_result.get("arbitration_history", [])
                    result["arbgraph_meta"]["rounds"] = len(history)

                    result["answer"] = generator.generate(
                        query=query,
                        context={"validated_claims": validated_claims},
                    )
                    result["arbgraph_meta"]["status"] = "success"

    except Exception as e:
        result["arbgraph_meta"]["status"] = "exception"
        result["arbgraph_meta"]["error_stage"] = "runtime"
        result["answer"] = f"Error: {str(e)}"

    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

    return result


def load_checkpoint(output_file: str) -> set:
    processed = set()
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    processed.add(json.loads(line)["id"])
                except Exception:
                    continue
    return processed


print("✓ Pipeline functions defined (exact copy from run_arbgraph.py)")

## Cell 9 — Run experiment

In [ ]:
from tqdm.auto import tqdm
import time

# ── Output directory ────────────────────────────────────────────────────────
output_file = CFG["output_file"]
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# ── Resume from checkpoint ──────────────────────────────────────────────────
processed_ids = load_checkpoint(output_file)
remaining = [q for q in questions if q["id"] not in processed_ids]

print(f"Total questions  : {len(questions)}")
print(f"Already done     : {len(processed_ids)}")
print(f"To process       : {len(remaining)}")
print(f"Output file      : {output_file}")
print()

# ── Main loop ───────────────────────────────────────────────────────────────
stats = {"success": 0, "no_documents": 0, "no_raw_claims": 0,
         "no_canonical_claims": 0, "exception": 0}

for q in tqdm(remaining, desc="ArbGraph"):
    t0 = time.time()

    result = process_single_question(
        question_data=q,
        components=components,
        output_file=output_file,
        max_docs=CFG["max_docs"],
    )

    status = result["arbgraph_meta"]["status"]
    stats[status] = stats.get(status, 0) + 1
    elapsed = time.time() - t0

    tqdm.write(
        f"  [{result['id']}] status={status} "
        f"validated={result['arbgraph_meta']['validated_claims']} "
        f"suppressed={result['arbgraph_meta']['suppressed_claims']} "
        f"rounds={result['arbgraph_meta']['rounds']} "
        f"({elapsed:.1f}s)"
    )

print()
print("=" * 50)
print("DONE")
print("=" * 50)
for k, v in stats.items():
    print(f"  {k:<25} {v}")
print(f"\nOutput: {output_file}")

## Cell 10 — Inspect results

In [ ]:
import json
import pandas as pd

results = []
with open(output_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            results.append(json.loads(line))

print(f"Total results: {len(results)}\n")

# Summary table
rows = []
for r in results:
    meta = r.get("arbgraph_meta", {})
    rows.append({
        "id":          r["id"],
        "status":      meta.get("status"),
        "retrieval":   meta.get("retrieval_status"),
        "validated":   meta.get("validated_claims", 0),
        "suppressed":  meta.get("suppressed_claims", 0),
        "rounds":      meta.get("rounds", 0),
        "answer_len":  len(r.get("answer", "")),
        "question":    r["question"][:60] + "...",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Print one full answer
print("\n" + "="*60)
print("Sample answer (first successful result):")
print("="*60)
for r in results:
    if r["arbgraph_meta"]["status"] == "success":
        print(f"Q: {r['question']}\n")
        print(r["answer"])
        break

## Cell 11 — Download output file

On Colab: downloads via browser.  
On Kaggle: file is already in `/kaggle/working/` — visible in the Output tab.

In [ ]:
try:
    from google.colab import files
    files.download(output_file)
    print("✓ Download started (Colab)")
except ImportError:
    print(f"Kaggle: output is at {output_file}")
    print("It will appear in the Output tab when the notebook finishes.")

---
## Appendix — Kaggle path adjustments

When running on **Kaggle**, change these two lines in Cell 2 and Cell 5:

```python
# Cell 2
REPO_DIR = "/kaggle/working/ArbGraph"

# Cell 5 — update all paths in DATASET_PRESETS to use /kaggle/working/
# e.g.
"input_file":  "/kaggle/input/my-dataset/longfact_test.json",
"output_file": "/kaggle/working/outputs/longfact/arbgraph_predictions.jsonl",
```

Upload your dataset file as a Kaggle dataset and attach it to the notebook, then reference it under `/kaggle/input/<dataset-name>/`.

## Supported dataset formats — quick reference

| Format | Example |
|--------|--------|
| **JSON list** | `[{"id":"1", "Question":"..."}]` |
| **JSON nested** | `{"data": [{"question_id":"1", "question":"..."}]}` — set `data_path: "data"` |
| **JSONL** | one JSON object per line |
| **CSV** | header row with `id` and `question` columns |
| **TSV** | same but tab-separated |

All formats are normalised to `{"id": ..., "Question": ...}` before entering the pipeline.